# 🚀 Spaceship Titanic — Exploratory Data Analysis

**Goal:** Predict which passengers were transported to an alternate dimension.

Dataset: [Kaggle Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='mako')
plt.rcParams['figure.dpi'] = 120

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')

## 1. Dataset Overview

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## 2. Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
train['Transported'].value_counts().plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452'])
ax.set_title('Target Distribution', fontsize=14)
ax.set_xticklabels(['Not Transported', 'Transported'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x()+p.get_width()/2, p.get_height()+30), ha='center')
plt.tight_layout()
plt.show()
print(train['Transported'].value_counts(normalize=True).round(3))

## 3. Missing Values

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, 4))
missing.plot(kind='bar', ax=ax, color='#c0392b')
ax.set_title('Missing Values per Feature', fontsize=14)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Numerical Feature Distributions

In [ ]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), num_cols):
    for transported, grp in train.groupby('Transported'):
        grp[col].dropna().plot(kind='hist', bins=40, alpha=0.6, ax=ax,
                               label='Transported' if transported else 'Not Transported')
    ax.set_title(col, fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Numerical Features by Target', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 5. Categorical Features

In [ ]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), cat_cols):
    ct = pd.crosstab(train[col], train['Transported'], normalize='index')
    ct.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452'], width=0.7)
    ax.set_title(f'{col} → Transport Rate', fontsize=12)
    ax.set_ylabel('Proportion')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(['Not Transported','Transported'], fontsize=8)

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_df = train[num_cols + ['Transported']].copy()
corr_df['Transported'] = corr_df['Transported'].astype(int)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df.corr()))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', mask=mask, cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Cabin Analysis

In [ ]:
cabin = train['Cabin'].str.split('/', expand=True)
train['Deck'] = cabin[0]
train['Side'] = cabin[2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ['Deck', 'Side']):
    ct = pd.crosstab(train[col], train['Transported'], normalize='index')
    ct[True].sort_values().plot(kind='barh', ax=ax, color='#2ecc71')
    ax.set_title(f'Transport Rate by {col}', fontsize=13)
    ax.set_xlabel('Transport Rate')
    ax.axvline(0.5, color='red', linestyle='--', label='50%')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Spending Insights

In [ ]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
train['TotalSpend'] = train[spend_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log spend by target
for transported, grp in train.groupby('Transported'):
    np.log1p(grp['TotalSpend']).plot(kind='hist', bins=50, alpha=0.6, ax=axes[0],
                                      label='Transported' if transported else 'Not')
axes[0].set_title('Log(Total Spend) by Target')
axes[0].legend()

# Zero spend vs transport
train['ZeroSpend'] = (train['TotalSpend'] == 0)
ct = pd.crosstab(train['ZeroSpend'], train['Transported'], normalize='index')
ct.plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'])
axes[1].set_title('Transport Rate: Zero vs Non-Zero Spend')
axes[1].set_xticklabels(['Has Spend','Zero Spend'], rotation=0)

plt.tight_layout()
plt.show()

## 9. Key Findings Summary

| Feature | Observation |
|---------|------------|
| **CryoSleep** | Passengers in cryo sleep are ~82% more likely to be transported |
| **Total Spend** | Zero spend strongly correlates with being transported |
| **Deck** | Deck B & C passengers have higher transport rates |
| **HomePlanet** | Europa passengers more likely to be transported |
| **Age** | Young children (< 12) have notably different transport rates |
| **VIP** | VIP status slightly reduces transport probability |